# Bookly Customer Support Agent — Live Demo

**Decagon Solutions Engineering Take-Home**

**Thesis:** Great CX agents are *grounded, clarify-first, and action-capable*. They never hallucinate transactional data — they ask focused questions, then act via tools.

This notebook uses the same orchestrator as the web UI (`agent/orchestrator.py`).

### Demo data (15 orders — ORD-1001 through ORD-1015)
| Order ID | Email | Status |
|----------|-------|--------|
| ORD-1001 | alice@example.com | Shipped |
| ORD-1006 | frank@example.com | Delayed |
| ORD-1007 | grace@example.com | Out for delivery |
| ORD-1008 | henry@example.com | Cancelled |
| ORD-1010 | jack@example.com | Delivered |

See `data/mock_orders.json` for the full dataset.

## Step 1: Bootstrap (run first)

**Note:** Colab only downloads this notebook file — Step 1 auto-clones the full repo from GitHub.

| Environment | What this cell does |
|-------------|---------------------|
| **Google Colab** | Clones `andrewwchild/booklydemo`, installs deps, loads API key |
| **Local / Cursor** | Finds the project root on disk |

**Optional:** Add Colab Secret `OPENAI_API_KEY` via the 🔑 sidebar for live LLM mode.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

# Default repo — Colab only downloads the notebook, not the full project
DEFAULT_REPO = "https://github.com/andrewwchild/booklydemo.git"
REPO_URL = DEFAULT_REPO  # override only if using a fork

IN_COLAB = "google.colab" in sys.modules


def _find_project_root() -> Path | None:
    for start in [Path.cwd(), *Path.cwd().parents]:
        path = start
        for _ in range(6):
            if (path / "agent").is_dir() and (path / "requirements.txt").exists():
                return path
            if path.parent == path:
                break
            path = path.parent
    return None


if IN_COLAB:
    root = _find_project_root()

    if root is None:
        clone_url = REPO_URL or os.environ.get("BOOKLY_REPO_URL", DEFAULT_REPO)
        clone_dir = Path("/content/booklydemo")
        if clone_dir.exists():
            print("✓ Repo already cloned — reusing /content/booklydemo")
            os.chdir(clone_dir)
            root = Path.cwd()
        else:
            print(f"Cloning {clone_url} ...")
            subprocess.run(["git", "clone", clone_url, str(clone_dir)], check=True)
            os.chdir(clone_dir)
            root = Path.cwd()

    os.chdir(root)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)

    try:
        from google.colab import userdata
        os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
        print("✓ OpenAI API key loaded from Colab Secrets")
    except Exception:
        print("○ No OPENAI_API_KEY secret — mock mode (still works for demo)")

    ROOT = Path.cwd()
    print(f"✓ Colab ready — project root: {ROOT}")

else:
    candidates = [Path.cwd().resolve()]
    ipynb = globals().get("__vsc_ipynb_file__")
    if ipynb:
        candidates.insert(0, Path(ipynb).resolve().parent)

    ROOT = None
    for start in candidates:
        path = start
        for _ in range(8):
            if (path / "agent").is_dir() and (path / "requirements.txt").exists():
                ROOT = path
                break
            if path.parent == path:
                break
            path = path.parent
        if ROOT:
            break

    if ROOT is None:
        raise RuntimeError("Could not find project root. Try: %cd ~/Projects/bookly-cs-agent")

    print(f"✓ Local ready — project root: {ROOT}")

## Step 2: Initialize agent

In [ ]:
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from dotenv import load_dotenv
load_dotenv(ROOT / ".env")

from agent.demo_helpers import new_session, print_turn
from agent.orchestrator import BooklyAgent
from agent.prompts import SYSTEM_PROMPT
from tools.registry import TOOL_DEFINITIONS

agent, memory = new_session(BooklyAgent())
mode = "live (OpenAI)" if not agent.use_mock else "mock (no API key)"
print(f"Agent ready — mode: {mode}")
if not agent.use_mock:
    print(f"Model: {agent.model}")

In [ ]:
def chat(message: str, *, show_tools: bool = True) -> None:
    """Send one message and print the agent response (+ tool calls)."""
    result = agent.chat(memory, message)
    print_turn(message, result, show_tools=show_tools)


def reset() -> None:
    """Start a fresh conversation (use between demo scenarios)."""
    global agent, memory
    agent, memory = new_session(BooklyAgent())
    print("Session reset.")

---
## Demo 1: Clarifying question + tool use (order status)

**Requirement:** Agent asks a clarifying question before acting, then uses a tool.

**Say aloud:** *"The customer didn't give an order ID — so the agent clarifies first instead of guessing."*

In [ ]:
reset()
chat("Where is my order?")

In [ ]:
chat("ORD-1001")

---
## Demo 2: Multi-turn refund + action

**Requirement:** Multi-turn slot filling, then the agent takes an action via `initiate_refund`.

**Say aloud:** *"Refund needs order ID, reason, and email — the agent collects these across turns before calling the tool."*

In [ ]:
reset()
chat("I want a refund")

In [ ]:
chat("ORD-1003")

In [ ]:
chat("damaged book")

In [ ]:
chat("carol@example.com")

---
## Demo 3: Policy FAQ (grounded answers)

**Say aloud:** *"For policies, the agent pulls official text — it doesn't invent return windows."*

In [ ]:
reset()
chat("What's your return policy?")

---
## Architecture peek (for Slide 2)

Show these when explaining how inquiries flow through the system.

In [ ]:
print("Tools registered with the LLM:")
for tool in TOOL_DEFINITIONS:
    fn = tool["function"]
    print(f"  • {fn['name']}: {fn['description'][:80]}...")

In [ ]:
print("System prompt (excerpt):")
print(SYSTEM_PROMPT[:600], "\n...")

---
## Free-form (optional)

Type anything live — useful if the interviewer throws a curveball.

---
## Demo 4: Stock check (inventory tool)

**Say aloud:** *"For product questions, the agent checks the catalog — it doesn't guess availability."*

In [ ]:
reset()
chat("Is Fourth Wing in stock?")

In [ ]:
# Edit the message below, then run this cell
chat("I can't log in to my account")